# 00 - Project Setup and Corpus Audit

                This notebook creates the reviewer-facing project structure, cleans the article table, writes the corpus audit, and exports the basic descriptive trend figures. It deliberately does not exclude reviews, editorials, bibliographies, anniversary items, or short-text records silently; it flags them so core and extended analyses can be compared later.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from hsr_analysis.common import *

import numpy as np
import pandas as pd

ensure_project_structure(ROOT)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

print(f"Project root: {ROOT}")

## Load and Clean the Corpus

In [ ]:
raw, source_path = load_article_source(ROOT)
print(f"Loaded {len(raw):,} rows from {source_path.relative_to(ROOT)}")

articles = build_articles_clean(raw)

exclusions_path = ROOT / "data/article_exclusions.csv"
if exclusions_path.exists():
    exclusions = pd.read_csv(exclusions_path)
    if not exclusions.empty and "article_id" in exclusions.columns:
        exclusions = exclusions.set_index("article_id")
        for col, target in [
            ("exclude_from_core", "corpus_inclusion_core"),
            ("exclude_from_extended", "corpus_inclusion_extended"),
        ]:
            if col in exclusions.columns:
                ids = exclusions.index[exclusions[col].fillna(False).astype(bool)]
                articles.loc[articles["article_id"].isin(ids), target] = False

write_csv(articles, ROOT / "outputs/tables/articles_clean.csv")
print(articles.shape)
display(articles.head(3))

## Corpus Audit

In [ ]:
def missing_count(series):
    return int(series.map(lambda x: compact_ws(x) == "").sum())

metrics = [
    ("n_records_total_raw", len(raw)),
    ("n_records_valid_year", len(articles)),
    ("n_unique_titles", articles["title"].map(compact_ws).str.lower().nunique()),
    ("n_duplicate_titles", int(articles["title"].map(compact_ws).str.lower().duplicated().sum())),
    ("n_missing_year_raw", int(pd.to_numeric(raw.get("year"), errors="coerce").isna().sum())),
    ("n_missing_authors", missing_count(articles["authors"])),
    ("n_missing_abstract_en", missing_count(articles["abstract_en"])),
    ("n_missing_abstract_de", missing_count(articles["abstract_de"])),
    ("n_missing_full_text", missing_count(articles["full_text"])),
    ("n_sufficient_text", int(articles["has_sufficient_text"].sum())),
    ("n_core_inclusion", int(articles["corpus_inclusion_core"].sum())),
    ("n_extended_inclusion", int(articles["corpus_inclusion_extended"].sum())),
    ("n_potential_reviews", int(articles["is_review"].sum())),
    ("n_potential_editorials_or_introductions", int(articles["is_editorial_or_intro"].sum())),
    ("n_potential_bibliographies", int(articles["is_bibliography"].sum())),
    ("n_potential_autobiographical_or_anniversary_items", int(articles["is_autobiographical_or_anniversary"].sum())),
]
audit = pd.DataFrame(metrics, columns=["metric", "value"])
write_csv(audit, ROOT / "outputs/tables/corpus_audit.csv")

by_year = articles.groupby("year").size().reset_index(name="n_articles")
by_language = articles.groupby("language", dropna=False).size().reset_index(name="n_articles")
by_journal = articles.groupby("journal", dropna=False).size().reset_index(name="n_articles")
by_type = articles.groupby("type_en", dropna=False).size().reset_index(name="n_articles")
by_period = articles.groupby("period", dropna=False).size().reset_index(name="n_articles")

write_csv(by_year, ROOT / "outputs/tables/corpus_by_year.csv")
write_csv(by_language, ROOT / "outputs/tables/corpus_by_language.csv")
write_csv(by_journal, ROOT / "outputs/tables/corpus_by_journal_or_supplement.csv")
write_csv(by_type, ROOT / "outputs/tables/corpus_by_type.csv")
write_csv(by_period, ROOT / "outputs/tables/corpus_by_period.csv")

audit_md = [
    "# HSR Corpus Audit",
    "",
    f"Source file: `{source_path.relative_to(ROOT)}`",
    "",
    "## Core Counts",
    "",
]
audit_md.extend(f"- `{row.metric}`: {row.value}" for row in audit.itertuples())
audit_md.extend(
    [
        "",
        "## Reviewer-Relevant Treatment",
        "",
        "- The modelling corpus is not filtered until each document has been assigned text, language, type, and uncertainty flags.",
        "- Reviews, editorials, introductions, bibliographies, supplements, and anniversary/autobiographical items are retained as flagged records.",
        "- Core and extended inclusion flags are written to `outputs/tables/articles_clean.csv` and can be toggled in later notebooks.",
        "- UMAP coordinates are used for visualization only; quantitative proximity is calculated in the embedding space or derived graph.",
    ]
)
write_markdown("\n".join(audit_md), ROOT / "outputs/diagnostics/corpus_audit.md")
display(audit)

## Basic Trends

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

articles["n_authors"] = articles["authors"].map(lambda x: len(split_author_string(x)))

basic_year = (
    articles.groupby("year")
    .agg(
        n_articles=("article_id", "nunique"),
        n_core_articles=("corpus_inclusion_core", "sum"),
        mean_authors=("n_authors", "mean"),
        share_english=("language", lambda x: (x == "en").mean()),
        share_german=("language", lambda x: (x == "de").mean()),
        share_reviews=("is_review", "mean"),
        share_editorials_or_intros=("is_editorial_or_intro", "mean"),
        share_bibliographies=("is_bibliography", "mean"),
    )
    .reset_index()
)
basic_period = (
    articles.groupby("period")
    .agg(
        n_articles=("article_id", "nunique"),
        n_core_articles=("corpus_inclusion_core", "sum"),
        mean_authors=("n_authors", "mean"),
        share_english=("language", lambda x: (x == "en").mean()),
        share_german=("language", lambda x: (x == "de").mean()),
    )
    .reset_index()
)
write_csv(basic_year, ROOT / "outputs/tables/basic_trends_by_year.csv")
write_csv(basic_period, ROOT / "outputs/tables/basic_trends_by_period.csv")

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(basic_year["year"], basic_year["n_articles"], color="#4c78a8")
ax.set_title("Articles per year")
ax.set_xlabel("Year")
ax.set_ylabel("Number of articles")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_01_articles_per_year.png", dpi=220)
save_caption(ROOT, "fig_01_articles_per_year.png", "Annual article counts in the cleaned HSR corpus.")
plt.show()

lang_year = articles.pivot_table(index="year", columns="language", values="article_id", aggfunc="nunique", fill_value=0)
lang_share = lang_year.div(lang_year.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(11, 4))
lang_share[[c for c in ["en", "de", "fr"] if c in lang_share.columns]].plot.area(ax=ax, alpha=0.8)
ax.set_title("Language share over time")
ax.set_xlabel("Year")
ax.set_ylabel("Share of articles")
ax.legend(title="Language", loc="upper left")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_02_language_share_over_time.png", dpi=220)
save_caption(ROOT, "fig_02_language_share_over_time.png", "Share of HSR articles by publication language over time.")
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(basic_year["year"], basic_year["mean_authors"], color="#f58518", marker="o", markersize=3)
ax.set_title("Authorship size over time")
ax.set_xlabel("Year")
ax.set_ylabel("Mean number of authors")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_03_authorship_size_over_time.png", dpi=220)
save_caption(ROOT, "fig_03_authorship_size_over_time.png", "Mean number of listed authors per article by year.")
plt.show()

doc_flags = articles.melt(
    id_vars=["year", "article_id"],
    value_vars=["is_review", "is_editorial_or_intro", "is_bibliography", "is_autobiographical_or_anniversary"],
    var_name="document_flag",
    value_name="flagged",
)
doc_year = doc_flags.groupby(["year", "document_flag"])["flagged"].mean().reset_index()
fig, ax = plt.subplots(figsize=(11, 4))
sns.lineplot(data=doc_year, x="year", y="flagged", hue="document_flag", ax=ax)
ax.set_title("Flagged document types over time")
ax.set_xlabel("Year")
ax.set_ylabel("Share of articles")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_04_document_type_over_time.png", dpi=220)
save_caption(ROOT, "fig_04_document_type_over_time.png", "Share of records flagged as reviews, editorials/introductions, bibliographies, or anniversary/autobiographical items.")
plt.show()